# CNC Guard — Verificación del entorno

Este notebook comprueba el **setup académico**. No es la entrega terminada, no evalúa la capacidad predictiva de un modelo y no se conecta a una máquina.

Antes de ejecutarlo, seguir el Markdown de instalación: una persona genera `.python-version` y `uv.lock`, se sincronizan las dependencias y se registra el kernel **Python (CNC Guard / uv)**. Este archivo no instala paquetes ni descarga datos.

Ejecutar mediante **Restart Kernel → Run All**. Cualquier comprobación fallida debe corregirse antes de continuar con el análisis del reto.

In [1]:
from cnc_guard.runtime import configure_resources, project_root

ROOT = project_root()
SETTINGS = configure_resources(ROOT)

print("Proyecto localizado y configuración de recursos cargada.")
print("Hilos BLAS propuestos:", SETTINGS["resources"]["blas_threads"])
print("Trabajos por modelo propuestos:", SETTINGS["resources"]["model_jobs"])

Proyecto localizado y configuración de recursos cargada.
Hilos BLAS propuestos: 1
Trabajos por modelo propuestos: 2


## 1. Identidad del intérprete y bloqueo

Se exige el parche de CPython acordado y un entorno aislado. La comprobación no muestra rutas personales ni secretos. La existencia de un lock no demuestra por sí sola una instalación correcta; las pruebas siguientes verifican parte de la interoperabilidad.

In [2]:
import hashlib
import platform
import sys

pin_file = ROOT / ".python-version"
lock_file = ROOT / "uv.lock"
if not pin_file.is_file() or not lock_file.is_file():
    raise RuntimeError("Falta el pin de Python o uv.lock. Completa el setup de referencia.")
if sys.prefix == sys.base_prefix:
    raise RuntimeError("El kernel no está dentro de un entorno virtual del proyecto.")
expected_python = pin_file.read_text(encoding="utf-8").strip()
if platform.python_version() != expected_python:
    raise RuntimeError("Python no coincide con el parche acordado por el equipo.")
if platform.python_implementation() != "CPython" or sys.maxsize <= 2**32:
    raise RuntimeError("Este perfil requiere CPython estándar de 64 bits.")

print("Python:", platform.python_version())
print("Sistema:", platform.system(), platform.machine())
print("SHA-256 de uv.lock:", hashlib.sha256(lock_file.read_bytes()).hexdigest())

Python: 3.13.13
Sistema: Linux x86_64
SHA-256 de uv.lock: 6fef1cb4751a0683c1b9ec0c04c3222af6253d5d6f4af68b31e3e03229cc950d


## 2. Prueba de interoperabilidad

El script comprueba importaciones, pandas/Parquet/DuckDB, dos estimadores pequeños, un modelo que él mismo genera y un gráfico PNG. Usa únicamente datos numéricos de juguete. El proceso hijo utiliza **este mismo intérprete** y hereda la configuración de hilos.

La prueba escribe `reports/environment.json`; sus temporales se eliminan al terminar. `smoke_seconds` mide esta comprobación y `rss_end_mib_not_peak` es la memoria RSS final, no la memoria máxima. Ninguna de ambas cifras es un benchmark del reto completo.

In [3]:
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "verify_setup.py")],
    cwd=ROOT,
    capture_output=True,
    text=True,
    timeout=300,
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(
        "La comprobación del setup ha fallado.\n" + result.stdout + "\n" + result.stderr
    )
print(result.stdout)

{
  "purpose": "Prueba de instalación; no validación predictiva ni industrial",
  "python": "3.13.13",
  "implementation": "CPython",
  "os": "Linux",
  "machine": "x86_64",
  "uv": "uv 0.12.15 (d35f1f270 2026-09-15 x86_64-unknown-linux-gnu)",
  "versions": {
    "numpy": "2.5.3",
    "pandas": "2.3.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.9.1",
    "duckdb": "1.5.5",
    "pyarrow": "25.0.1",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
    "jupyterlab": "4.6.3",
    "ipykernel": "7.3.0",
    "nbconvert": "7.17.1",
    "nbformat": "5.11.1",
    "pytest": "9.1.1",
    "psutil": "7.2.2",
    "ruff": "0.16.8"
  },
  "uv_lock_sha256": "6fef1cb4751a0683c1b9ec0c04c3222af6253d5d6f4af68b31e3e03229cc950d",
  "smoke_seconds": 9.401,
  "rss_end_mib_not_peak": 373.78,
  "status": "passed"
}



## 3. Resultado y alcance

La siguiente celda confirma el resultado recién generado y que pertenece al lock actual. No reemplaza las pruebas `pytest`, la revisión de vulnerabilidades, la validación del notebook final ni la aceptación del piloto industrial.

In [4]:
import json

report = json.loads((ROOT / "reports" / "environment.json").read_text(encoding="utf-8"))
if report.get("status") != "passed":
    raise RuntimeError("El informe no registra una verificación correcta.")
if report["uv_lock_sha256"] != hashlib.sha256(lock_file.read_bytes()).hexdigest():
    raise RuntimeError("El informe no corresponde al lock actual.")

print("Verificación del entorno completada.")
print("Pendiente: desarrollar y evaluar CNC_Guard.ipynb con los datos del reto.")

Verificación del entorno completada.
Pendiente: desarrollar y evaluar CNC_Guard.ipynb con los datos del reto.
